In [37]:
import os
import sys

from pathlib import Path

PROJECT_ROOT = Path("/workspace")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ai_sac.connectors.sac.auth import get_access_token, SacAuthError

In [38]:
try:
    token = get_access_token()

    print("OAuth authentication: SUCCESS")
    print(f"Token received: {bool(token)}")
    print(f"Token length: {len(token)}")

except SacAuthError as error:
    print(f"OAuth authentication: FAILED")
    print(error)

OAuth authentication: SUCCESS
Token received: True
Token length: 1957


<h3>Get Fact Data Model<h3>

In [39]:
from ai_sac.connectors.sac.client import get_fact_data
import pandas as pd
from pathlib import Path
import time

start_time = time.time()

result = get_fact_data(
    params={"pagesize": 1000},
    max_pages=500,
)

elapsed = time.time() - start_time

print("Extraction complete:", result["is_complete"])
print("Pages fetched:", result["pages_fetched"])
print("Total records:", len(result["records"]))
print("Has more data:", result["has_more_data"])
print(f"Elapsed time: {elapsed:.2f} seconds")

Extraction complete: True
Pages fetched: 33
Total records: 32225
Has more data: False
Elapsed time: 40.95 seconds


In [40]:
if not result["is_complete"]:
    raise RuntimeError(
        "Extraction belum lengkap. Jangan simpan sebagai dataset final."
    )

fact_df = pd.DataFrame(result["records"])

measure_columns = [
    "Amount",
    "MTDAmount",
    "ProgCM",
    "ProgNM",
]

for col in measure_columns:
    fact_df[col] = pd.to_numeric(
        fact_df[col],
        errors="coerce",
    )

print("Shape:", fact_df.shape)
print("\nData types:")
print(fact_df.dtypes)

display(fact_df.head())

Shape: (32225, 10)

Data types:
Version             str
Period              str
ProfitCenter        str
CompanyCode         str
Account             str
CostCenter          str
Amount          float64
MTDAmount       float64
ProgCM          float64
ProgNM          float64
dtype: object


,Version,Period,ProfitCenter,CompanyCode,Account,CostCenter,Amount,MTDAmount,ProgCM,ProgNM
0,public.Actual,202012,#,1000,LM_ASL_02_CutOff,#,2.650000e+11,NaN,NaN,NaN
1,public.Actual,202012,#,1000,LM_ASL_03_CutOff,#,1.530000e+11,NaN,NaN,NaN
2,public.Actual,202012,#,1000,LM_ASL_04_CutOff,#,1.900000e+10,NaN,NaN,NaN
3,public.Actual,202012,#,1000,LM_ASL_05_CutOff,#,3.590000e+10,NaN,NaN,NaN
4,public.Actual,202012,#,1000,LM_ASL_06_CutOff,#,3.290000e+10,NaN,NaN,NaN


In [30]:
print("Measure completeness:")
display(fact_df[measure_columns].notna().sum())

print("\nVersion distribution:")
display(fact_df["Version"].value_counts(dropna=False))

print("\nPeriod range:")
print(
    fact_df["Period"].min(),
    "to",
    fact_df["Period"].max(),
)

print("\nDuplicate key combinations:")
print(
    fact_df.duplicated(
        subset=[
            "Version",
            "Period",
            "ProfitCenter",
            "CompanyCode",
            "Account",
            "CostCenter",
        ]
    ).sum()
)

Measure completeness:


Amount       31682
MTDAmount     5760
ProgCM         208
ProgNM           0
dtype: int64


Version distribution:


Version
public.RKAP        23827
public.Actual       8190
public.Prognosa      208
Name: count, dtype: int64


Period range:
202012 to 202806

Duplicate key combinations:
0


In [52]:
output_dir = Path("/workspace/data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "fact_data.parquet"

fact_df.to_parquet(
    output_file,
    engine="pyarrow",
    index=False,
)

print("Saved:", output_file)
print(f"File size: {output_file.stat().st_size / 1024**2:.2f} MB")
print(Path)

Saved: /workspace/data/raw/fact_data.parquet
File size: 0.05 MB
<class 'pathlib.Path'>


In [ ]:
from ai_sac.connectors.sac.client import get_providers, SacApiError

try:
    providers = get_providers()

    print(f"Total providers found: {len(providers)}")

    for provider in providers:
        print(
            provider.get("ProviderName"),
            "|",
            provider.get("ProviderID"),
        )

except SacApiError as error:
    print("SAC API FAILED")
    print(error)

In [35]:
import pandas as pd
from pathlib import Path

parquet_file = Path("data/raw/fact_data.parquet")

# Baca ulang file dari disk
fact_parquet_df = pd.read_parquet(parquet_file)

print("Original shape:", fact_df.shape)
print("Parquet shape:", fact_parquet_df.shape)

print("Same columns:",
      fact_df.columns.tolist() == fact_parquet_df.columns.tolist())

print("Same row count:",
      len(fact_df) == len(fact_parquet_df))

print("Same data:",
      fact_df.equals(fact_parquet_df))

print("File size:",
      f"{parquet_file.stat().st_size / 1024**2:.2f} MB")

Original shape: (32225, 10)
Parquet shape: (32225, 10)
Same columns: True
Same row count: True
Same data: True
File size: 0.05 MB


In [47]:
from pathlib import Path

project_root = Path("/workspace")

parquet_path = (
    project_root / "notebooks" / "data" / "raw" / "fact_data.parquet"
)

db_path = (
    project_root / "data" / "processed" / "ai_sac.duckdb"
)

print("Parquet exists:", parquet_path.exists())
print("Parquet path:", parquet_path)

Parquet exists: True
Parquet path: /workspace/notebooks/data/raw/fact_data.parquet


In [51]:
from pathlib import Path
import shutil

source = Path("/workspace/notebooks/data/raw/fact_data.parquet")
target = Path("/workspace/data/raw/fact_data.parquet")

# Pastikan file sumber tersedia
assert source.exists(), f"File tidak ditemukan: {source}"

# Pastikan folder tujuan tersedia
target.parent.mkdir(parents=True, exist_ok=True)

# Pindahkan file
shutil.move(str(source), str(target))

# Verifikasi
print("Source exists:", source.exists())
print("Target exists:", target.exists())
print("File size:", f"{target.stat().st_size / (1024**2):.2f} MB")

Source exists: False
Target exists: True
File size: 0.05 MB
